In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import pipeline, metrics, preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import mean_gamma_deviance, mean_poisson_deviance

from interpret import perf, show
from interpret.glassbox import ExplainableBoostingRegressor


import kagglehub
import os

### Load datasets

In [ ]:

path = kagglehub.dataset_download("karansarpal/fremtpl2-french-motor-tpl-insurance-claims")

os.listdir(path)


freq = pd.read_csv(os.path.join(path, "freMTPL2freq.csv"))
sev = pd.read_csv(os.path.join(path, "freMTPL2sev.csv"))



100%|██████████| 6.94M/6.94M [00:00<00:00, 54.1MB/s]

Extracting files...


In [ ]:
#merge

data = freq.merge(sev, on="IDpol", how="left")
data["ClaimAmount"] = data["ClaimAmount"].fillna(0)

data

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,ClaimAmount
0,1.0,1,0.10000,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,0.0
1,3.0,1,0.77000,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes,0.0
2,5.0,1,0.75000,6,2,52,50,B12,Diesel,B,54,Picardie,0.0
3,10.0,1,0.09000,7,0,46,50,B12,Diesel,B,76,Aquitaine,0.0
4,11.0,1,0.84000,7,0,46,50,B12,Diesel,B,76,Aquitaine,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679508,6114326.0,0,0.00274,4,0,54,50,B12,Regular,E,3317,Provence-Alpes-Cotes-D'Azur,0.0
679509,6114327.0,0,0.00274,4,0,41,95,B12,Regular,E,9850,Ile-de-France,0.0
679510,6114328.0,0,0.00274,6,2,45,50,B12,Diesel,D,1323,Rhone-Alpes,0.0
679511,6114329.0,0,0.00274,4,0,60,50,B12,Regular,B,95,Bourgogne,0.0


## Claim frequency

In [ ]:

df_freq = data.copy()

# non sto usando tutto il dataset perchè mi ci stava mettendo troppo, però poi lo possiamo anche usare tutto
sample_size = 200000

df_sample = df_freq.sample(n=sample_size, random_state=42)

y_freq = df_sample["ClaimNb"]
X_freq = df_sample.drop(columns=["IDpol", "ClaimNb", "ClaimAmount"])

In [ ]:

X_train_freq, X_test_freq, y_train_freq, y_test_freq = train_test_split(
    X_freq,
    y_freq,
    test_size=0.2,
    random_state=1
)


train_freq = pd.concat([X_train_freq, y_train_freq], axis=1)
test_freq = pd.concat([X_test_freq, y_test_freq], axis=1)



(160000, 10) (40000, 10)


In [ ]:
import numpy as np

# Offset vectors, da quanto ho capito servono perchè un tizio che ha fatto un incidente in 2 mesi sarebbe altrimenti uguale a un tizio che ne ha fatto 1 in 1 anno
train_expo_freq = X_train_freq["Exposure"]
test_expo_freq = X_test_freq["Exposure"]

# Log transformation (offset)
expo_init_train_freq = np.log(train_expo_freq)
expo_init_test_freq = np.log(test_expo_freq)

In [ ]:
X_train_freq = X_train_freq.drop(columns=["Exposure"])
X_test_freq = X_test_freq.drop(columns=["Exposure"])

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import mean_poisson_deviance
import numpy as np

def edr_poisson(ytrue, ypred, nul_pred, weights):

    nul_dev = mean_poisson_deviance(
        y_true=ytrue,
        y_pred=nul_pred,
        sample_weight=weights
    )

    mod_dev = mean_poisson_deviance(
        y_true=ytrue,
        y_pred=ypred,
        sample_weight=weights
    )

    res = 1 - mod_dev / nul_dev
    return res



n_train_freq = len(y_train_freq)
n_test_freq = len(y_test_freq)


train_weights_freq = np.ones(n_train_freq)
test_weights_freq = np.ones(n_test_freq)


lambda_null = y_train_freq.sum() / train_expo_freq.sum()


nul_pred_train_freq = lambda_null * train_expo_freq
nul_pred_test_freq = lambda_null * test_expo_freq


def gini(ytrue, ypred):
    n = len(ytrue)

    if n != len(ypred):
        raise ValueError("Vectors ytrue and ypred must have the same length")

    idx = np.argsort(ypred)

    y_sorted = np.array(ytrue)[idx]

    sum_left = np.sum((np.arange(1, n+1)) * y_sorted) / np.sum(ytrue)
    sum_right = np.sum(np.arange(n, 0, -1) / n)

    gini_value = sum_left - sum_right
    return gini_value



def normalized_gini(ytrue, ypred):
    gini_pred = gini(ytrue, ypred)

    gini_ref = gini(ytrue, ytrue)

    return gini_pred / gini_ref

#### The feature types are explicitly specified to ensure correct handling of numerical and categorical variables. Continuous variables are discretized using quantile-based binning, while categorical variables are treated as nominal features with separate score contributions for each level.

In [ ]:


feature_names = [
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "VehBrand",
    "VehGas",
    "Area",
    "Density",
    "Region"
]

feature_types = [
    "quantile",   # VehPower
    "quantile",   # VehAge
    "quantile",   # DrivAge
    "quantile",   # BonusMalus
    "nominal",    # VehBrand
    "nominal",    # VehGas
    "nominal",    # Area
    "quantile",   # Density
    "nominal"     # Region
]



In [ ]:
!pip install interpret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 63.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 30.7 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl siz

In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor

ebm_model_freq = ExplainableBoostingRegressor(
    feature_names=feature_names,
    feature_types=feature_types,
    objective="poisson_deviance"
)

non è stato fatto il grid search, ma si sono usati i parametri più o meno del paper

In [ ]:
ebm_model_freq = ExplainableBoostingRegressor(
    feature_names=feature_names,
    feature_types=feature_types,
    objective="poisson_deviance",
    interactions=5,
    learning_rate=0.01,
    smoothing_rounds=50,
    max_bins=128,
    outer_bags=4,
    random_state=42
)

In [ ]:
ebm_model_freq.fit(
    X_train_freq,
    y_train_freq,
    init_score=expo_init_train_freq
)

ExplainableBoostingRegressor(feature_names=['VehPower', 'VehAge', 'DrivAge',
                                            'BonusMalus', 'VehBrand', 'VehGas',
                                            'Area', 'Density', 'Region'],
                             feature_types=['quantile', 'quantile', 'quantile',
                                            'quantile', 'nominal', 'nominal',
                                            'nominal', 'quantile', 'nominal'],
                             interactions=5, learning_rate=0.01, max_bins=128,
                             objective='poisson_deviance', outer_bags=4,
                             smoothing_rounds=50)

In [ ]:
# Train predictions
tuned_predict_train = ebm_model_freq.predict(
    X_train_freq,
    init_score=expo_init_train_freq
)

# Test predictions
tuned_predict_test = ebm_model_freq.predict(
    X_test_freq,
    init_score=expo_init_test_freq
)

In [ ]:
RMSE_train_freq = np.sqrt(mean_squared_error(y_train_freq, tuned_predict_train))
MAE_train_freq = mean_absolute_error(y_train_freq, tuned_predict_train)

EDR_train_freq = edr_poisson(
    y_train_freq,
    tuned_predict_train,
    nul_pred_train_freq,
    train_weights_freq
)

Gini_train_freq = normalized_gini(y_train_freq, tuned_predict_train)

In [ ]:
RMSE_test_freq = np.sqrt(mean_squared_error(y_test_freq, tuned_predict_test))
MAE_test_freq = mean_absolute_error(y_test_freq, tuned_predict_test)

EDR_test_freq = edr_poisson(
    y_test_freq,
    tuned_predict_test,
    nul_pred_test_freq,
    test_weights_freq
)

Gini_test_freq = normalized_gini(y_test_freq, tuned_predict_test)

In [ ]:
results_freq = pd.DataFrame({
    "RMSE": [round(RMSE_train_freq,4), round(RMSE_test_freq,4)],
    "MAE": [round(MAE_train_freq,4), round(MAE_test_freq,4)],
    "EDR Poisson": [round(EDR_train_freq,4), round(EDR_test_freq,4)],
    "Gini_norm": [round(Gini_train_freq,4), round(Gini_test_freq,4)]
},
index=["D_train_freq", "D_test_freq"]
)

results_freq

,RMSE,MAE,EDR Poisson,Gini_norm
D_train_freq,0.2875,0.1058,0.0971,0.3868
D_test_freq,0.2754,0.1052,0.0803,0.3534


In [ ]:
from interpret import show


tuned_ebm_global = ebm_model_freq.explain_global()
fig_global_freq = tuned_ebm_global.visualize()

show(tuned_ebm_global)


3 casi: low, medium e high

In [ ]:
df_local = X_test_freq.copy()
df_local["pred"] = tuned_predict_test
df_local["actual"] = y_test_freq.values
df_sorted = df_local.sort_values("pred")

In [ ]:
n = len(df_sorted)

low_idx = df_sorted.index[int(0.05 * n)]
med_idx = df_sorted.index[int(0.50 * n)]
high_idx = df_sorted.index[int(0.95 * n)]

In [ ]:
sample_low = X_test_freq.loc[[low_idx]]
sample_med = X_test_freq.loc[[med_idx]]
sample_high = X_test_freq.loc[[high_idx]]

y_low = y_test_freq.loc[[low_idx]]
y_med = y_test_freq.loc[[med_idx]]
y_high = y_test_freq.loc[[high_idx]]

expo_low = expo_init_test_freq.loc[[low_idx]]
expo_med = expo_init_test_freq.loc[[med_idx]]
expo_high = expo_init_test_freq.loc[[high_idx]]

In [ ]:
local_low = ebm_model_freq.explain_local(
    sample_low,
    y_low,
    init_score=expo_low
)

local_med = ebm_model_freq.explain_local(
    sample_med,
    y_med,
    init_score=expo_med
)

local_high = ebm_model_freq.explain_local(
    sample_high,
    y_high,
    init_score=expo_high
)

In [ ]:

show(local_low)
show(local_med)
show(local_high)

### XGB

In [ ]:
!pip install xgboost

In [ ]:
import xgboost as xgb

X_train_xgb = X_train_freq.copy()
X_test_xgb = X_test_freq.copy()

# da quanto ho capito per xgb non vanno bene classificazioni ma lavora solo su variabili continue
for col in ["VehBrand", "VehGas", "Area", "Region"]:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

xgb_model = xgb.XGBRegressor(
    objective="count:poisson",
    enable_categorical=True,
    eval_metric="poisson-nloglik",
    learning_rate=0.1,
    n_estimators=200,
    max_depth=4,
    min_child_weight=1,
    gamma=0.05,
    reg_alpha=1,
    subsample=1,
    colsample_bytree=1,
    random_state=42
)




In [ ]:
xgb_model.fit(
    X_train_xgb,
    y_train_freq,
    sample_weight=train_expo_freq
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
             device=None, early_stopping_rounds=None, enable_categorical=True,
             eval_metric='poisson-nloglik', feature_types=None,
             feature_weights=None, gamma=0.05, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=1, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
pred_xgb_test = xgb_model.predict(X_test_xgb)
pred_xgb_train = xgb_model.predict(X_train_xgb)

RMSE_xgb_train = np.sqrt(mean_squared_error(y_train_freq, pred_xgb_train))
MAE_xgb_train = mean_absolute_error(y_train_freq, pred_xgb_train)

EDR_xgb_train = edr_poisson(
    y_train_freq,
    pred_xgb_train,
    nul_pred_train_freq,
    train_weights_freq
)

Gini_xgb_train = normalized_gini(y_train_freq, pred_xgb_train)

RMSE_xgb_test = np.sqrt(mean_squared_error(y_test_freq, pred_xgb_test))
MAE_xgb_test = mean_absolute_error(y_test_freq, pred_xgb_test)

EDR_xgb_test = edr_poisson(
    y_test_freq,
    pred_xgb_test,
    nul_pred_test_freq,
    test_weights_freq
)

Gini_xgb_test = normalized_gini(y_test_freq, pred_xgb_test)

In [ ]:
results_xgb_freq = pd.DataFrame({
    "RMSE": [round(RMSE_xgb_train,4), round(RMSE_xgb_test,4)],
    "MAE": [round(MAE_xgb_train,4), round(MAE_xgb_test,4)],
    "EDR Poisson": [round(EDR_xgb_train,4), round(EDR_xgb_test,4)],
    "Gini_norm": [round(Gini_xgb_train,4), round(Gini_xgb_test,4)]
},
index=["D_train_freq", "D_test_freq"]
)

results_xgb_freq



,RMSE,MAE,EDR Poisson,Gini_norm
D_train_freq,0.2777,0.1199,0.1132,0.3714
D_test_freq,0.2695,0.1197,0.0879,0.2971


In [ ]:
comparison_freq = pd.DataFrame({
    "Model": ["EBM", "XGB"],
    "RMSE": [round(RMSE_test_freq,4), round(RMSE_xgb_test,4)],
    "MAE": [round(MAE_test_freq,4), round(MAE_xgb_test,4)],
    "EDR": [round(EDR_test_freq,4), round(EDR_xgb_test,4)],
    "Gini": [round(Gini_test_freq,4), round(Gini_xgb_test,4)]
})

comparison_freq

,Model,RMSE,MAE,EDR,Gini
0,EBM,0.2754,0.1052,0.0803,0.3534
1,XGB,0.2695,0.1197,0.0879,0.2971


## SEVERITY

In [ ]:
severity_data = sev.merge(
    freq,
    on="IDpol",
    how="left"
).copy()

#credo che serva perchè il modella gamma può assumere solo valori positivi
severity_data = severity_data[
    severity_data["ClaimAmount"] > 0
].copy()

y_sev = severity_data["ClaimAmount"]

In [ ]:
feature_cols = [
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "VehBrand",
    "VehGas",
    "Area",
    "Density",
    "Region"
]

X_sev = severity_data[feature_cols]

In [ ]:
from sklearn.model_selection import train_test_split

X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
    X_sev,
    y_sev,
    test_size=0.2,
    random_state=1
)

anche qui non si è fatto grid search

In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor

ebm_sev = ExplainableBoostingRegressor(
    feature_names=feature_names,
    feature_types=feature_types,
    objective="gamma_deviance",
    interactions=0.9,
    learning_rate=0.02,
    smoothing_rounds=500,
    max_bins=32,
    max_interaction_bins = 32,
    max_leaves=3,
    outer_bags=14,
    random_state=42
)



In [ ]:

from sklearn.metrics import mean_gamma_deviance
import numpy as np


def edr_gamma(ytrue, ypred, nul_pred, weights):

    nul_dev = mean_gamma_deviance(y_true = ytrue, y_pred = nul_pred,
    sample_weight = weights)
    mod_dev = mean_gamma_deviance(y_true = ytrue, y_pred = ypred,
    sample_weight = weights)

    res = 1 - mod_dev / nul_dev
    return res


n_train_sev = len(y_train_sev)
n_test_sev = len(y_test_sev)

train_weights_sev = np.ones(n_train_sev)
test_weights_sev = np.ones(n_test_sev)


mean_sev_train = y_train_sev.mean()


nul_pred_train_sev = np.repeat(mean_sev_train, n_train_sev)
nul_pred_test_sev = np.repeat(mean_sev_train, n_test_sev)



def gini(ytrue, ypred):
    n = len(ytrue)

    if n != len(ypred):
        raise ValueError("Vectors ytrue and ypred must have the same length")

    idx = np.argsort(ypred)

    y_sorted = np.array(ytrue)[idx]

    sum_left = np.sum((np.arange(1, n+1)) * y_sorted) / np.sum(ytrue)
    sum_right = np.sum(np.arange(n, 0, -1) / n)

    gini_value = sum_left - sum_right
    return gini_value



def normalized_gini(ytrue, ypred):
    gini_pred = gini(ytrue, ypred)

    gini_ref = gini(ytrue, ytrue)

    return gini_pred / gini_ref

In [ ]:
ebm_sev.fit(X_train_sev, y_train_sev)

/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:871: UserWarning:

Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]



ExplainableBoostingRegressor(feature_names=['VehPower', 'VehAge', 'DrivAge',
                                            'BonusMalus', 'VehBrand', 'VehGas',
                                            'Area', 'Density', 'Region'],
                             feature_types=['quantile', 'quantile', 'quantile',
                                            'quantile', 'nominal', 'nominal',
                                            'nominal', 'quantile', 'nominal'],
                             interactions=0.9, learning_rate=0.02, max_bins=32,
                             max_interaction_bins=32, max_leaves=3,
                             objective='gamma_deviance')

In [ ]:
#  TRAIN
pred_train_sev = ebm_sev.predict(X_train_sev)

#  TEST
pred_test_sev = ebm_sev.predict(X_test_sev)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# TRAIN
RMSE_train_sev = np.sqrt(mean_squared_error(y_train_sev, pred_train_sev))
MAE_train_sev = mean_absolute_error(y_train_sev, pred_train_sev)

EDR_train_sev = edr_gamma(
    y_train_sev,
    pred_train_sev,
    nul_pred_train_sev,
    train_weights_sev
)

Gini_train_sev = normalized_gini(y_train_sev, pred_train_sev)


# TEST
RMSE_test_sev = np.sqrt(mean_squared_error(y_test_sev, pred_test_sev))
MAE_test_sev = mean_absolute_error(y_test_sev, pred_test_sev)

EDR_test_sev = edr_gamma(
    y_test_sev,
    pred_test_sev,
    nul_pred_test_sev,
    test_weights_sev
)

Gini_test_sev = normalized_gini(y_test_sev, pred_test_sev)

In [ ]:
results_sev = pd.DataFrame({
    "RMSE": [round(RMSE_train_sev,4), round(RMSE_test_sev,4)],
    "MAE": [round(MAE_train_sev,4), round(MAE_test_sev,4)],
    "EDR Gamma": [round(EDR_train_sev,4), round(EDR_test_sev,4)],
    "Gini_norm": [round(Gini_train_sev,4), round(Gini_test_sev,4)]
},
index=["D_train_sev", "D_test_sev"]
)

results_sev

,RMSE,MAE,EDR Gamma,Gini_norm
D_train_sev,12406.3755,1810.2298,0.0850,0.3737
D_test_sev,60598.2101,2949.3467,0.0428,0.3831


In [ ]:
from interpret import show


tuned_ebm_global = ebm_sev.explain_global()
fig_global_freq = tuned_ebm_global.visualize()

show(tuned_ebm_global)


### XGB

In [ ]:
import xgboost as xgb

X_train_sev_xgb = X_train_sev.copy()
X_test_sev_xgb = X_test_sev.copy()


for col in ["VehBrand", "VehGas", "Area", "Region"]:
    X_train_sev_xgb[col] = X_train_sev_xgb[col].astype("category")
    X_test_sev_xgb[col] = X_test_sev_xgb[col].astype("category")

xgb_sev = xgb.XGBRegressor(
    objective="reg:gamma",
    enable_categorical=True,
    eval_metric="gamma-nloglik",
    learning_rate=0.1,
    n_estimators=200,
    max_depth=4,
    min_child_weight=1,
    gamma=0,
    reg_alpha=1,
    subsample=1,
    colsample_bytree=0.75,
    random_state=42
)




In [ ]:
xgb_sev.fit(
    X_train_sev_xgb,
    y_train_sev
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.75, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='gamma-nloglik',
             feature_types=None, feature_weights=None, gamma=0,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
pred_sev_xgb_test = xgb_sev.predict(X_test_sev_xgb)
pred_sev_xgb_train = xgb_sev.predict(X_train_sev_xgb)

In [ ]:
RMSE_xgb_sev_test = np.sqrt(mean_squared_error(y_test_sev, pred_sev_xgb_test))
MAE_xgb_sev_test = mean_absolute_error(y_test_sev, pred_sev_xgb_test)

EDR_xgb_sev_test = edr_gamma(
    y_test_sev,
    pred_sev_xgb_test,
    nul_pred_test_sev,
    test_weights_sev
)

Gini_xgb_sev_test = normalized_gini(y_test_sev, pred_sev_xgb_test)

# TRAIN
RMSE_xgb_sev_train = np.sqrt(mean_squared_error(y_train_sev, pred_sev_xgb_train))
MAE_xgb_sev_train = mean_absolute_error(y_train_sev, pred_sev_xgb_train)

EDR_xgb_sev_train = edr_gamma(
    y_train_sev,
    pred_sev_xgb_train,
    nul_pred_train_sev,
    train_weights_sev
)

Gini_xgb_sev_train = normalized_gini(y_train_sev, pred_sev_xgb_train)

In [ ]:
results_xgb_sev = pd.DataFrame({
    "RMSE": [round(RMSE_xgb_sev_train,4), round(RMSE_xgb_sev_test,4)],
    "MAE": [round(MAE_xgb_sev_train,4), round(MAE_xgb_sev_test,4)],
    "EDR Gamma": [round(EDR_xgb_sev_train,4), round(EDR_xgb_sev_test,4)],
    "Gini_norm": [round(Gini_xgb_sev_train,4), round(Gini_xgb_sev_test,4)]
},
index=["D_train_sev", "D_test_sev"]
)

results_xgb_sev

,RMSE,MAE,EDR Gamma,Gini_norm
D_train_sev,11507.2234,1597.2309,0.2980,0.6101
D_test_sev,60604.2316,2816.7353,-0.0277,0.3661


In [ ]:
comparison_freq = pd.DataFrame({
    "Model": ["EBM", "XGB"],
    "RMSE": [round(RMSE_test_sev,4), round(RMSE_xgb_sev_test,4)],
    "MAE": [round(MAE_test_sev,4), round(MAE_xgb_sev_test,4)],
    "EDR": [round(EDR_test_sev,4), round(EDR_xgb_sev_test,4)],
    "Gini": [round(Gini_test_sev,4), round(Gini_xgb_sev_test,4)]
})

comparison_freq

,Model,RMSE,MAE,EDR,Gini
0,EBM,60598.2101,2949.3467,0.0428,0.3831
1,XGB,60604.2316,2816.7353,-0.0277,0.3661


credo ci sia overfitting di XGB